In [ ]:
# Keep exported joblib models compatible with the AI service runtime.
import importlib.util
import subprocess
import sys
if importlib.util.find_spec('google.colab') is not None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn==1.9.1', 'joblib==1.6.0'])


In [ ]:
import io
import json
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
try:
    from google.colab import files
except ModuleNotFoundError:
    files = None
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, recall_score
from sklearn.model_selection import cross_val_predict, StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


def find_ai_models_dir():
    for folder in (Path.cwd(), *Path.cwd().parents):
        if folder.name == 'ai-models':
            return folder
        if (folder / 'ai-models').is_dir():
            return folder / 'ai-models'
    raise FileNotFoundError('Cannot find ai-models/. Run this notebook from the project or ai-models folder.')


def read_csv_from_zip(zip_bytes):
    with zipfile.ZipFile(io.BytesIO(zip_bytes), 'r') as archive:
        csv_name = next(name for name in archive.namelist() if name.lower().endswith('.csv'))
        with archive.open(csv_name) as csv_file:
            return pd.read_csv(csv_file)


if files is not None:
    uploaded = files.upload()
    filename = next(iter(uploaded))
    if filename.lower().endswith('.zip'):
        df = read_csv_from_zip(uploaded[filename])
    else:
        df = pd.read_csv(io.BytesIO(uploaded[filename]))
    output_dir = Path.cwd()
else:
    ai_models_dir = find_ai_models_dir()
    csv_path = ai_models_dir / 'data' / 'diabetes.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
    else:
        archive_path = ai_models_dir / 'data' / 'archive.zip'
        if not archive_path.exists():
            raise FileNotFoundError(f'No diabetes.csv or archive.zip found in {archive_path.parent}')
        df = read_csv_from_zip(archive_path.read_bytes())
    output_dir = ai_models_dir / 'models'

output_dir.mkdir(parents=True, exist_ok=True)
feature_columns = [column for column in df.columns if column not in ('Outcome', 'index')]
X = df[feature_columns]
y = df['Outcome'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

invalid_zero_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
TARGET_SENSITIVITY = 0.80  # Illustrative screening assumption; confirm with clinical stakeholders.
MODEL_LABELS = {
    'logistic': 'Logistic Regression',
    'svm': 'Support Vector Machine (SVM)',
    'naive_bayes': 'Naive Bayes',
    'random_forest': 'Random Forest',
}

def make_pipeline(estimator):
    preprocessing = ColumnTransformer(
        [('median', SimpleImputer(strategy='median', missing_values=0), invalid_zero_columns)],
        remainder='passthrough',
    )
    return Pipeline([
        ('impute_invalid_zero', preprocessing),
        ('scaler', StandardScaler()),
        ('classifier', estimator),
    ])

def calculate_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true, predictions, average='weighted', zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(y_true, predictions)),
        'weighted_precision': float(weighted_precision),
        'weighted_recall': float(weighted_recall),
        'weighted_f1_score': float(weighted_f1),
        'positive_recall': float(recall_score(y_true, predictions, pos_label=1, zero_division=0)),
        'specificity': float(recall_score(y_true, predictions, pos_label=0, zero_division=0)),
        'predicted_positive_count': int(predictions.sum()),
    }

def choose_screening_threshold(y_true, probabilities):
    thresholds = np.unique(np.concatenate(([0.0, 1.0], probabilities)))
    feasible = []
    for threshold in thresholds:
        metrics = calculate_metrics(y_true, probabilities, float(threshold))
        if metrics['positive_recall'] >= TARGET_SENSITIVITY:
            feasible.append((metrics['specificity'], metrics['weighted_f1_score'], float(threshold), metrics))
    if not feasible:
        raise ValueError(f'No threshold reaches sensitivity target {TARGET_SENSITIVITY:.0%}')
    _, _, threshold, metrics = max(feasible)
    return threshold, metrics

models = {
    'logistic': make_pipeline(LogisticRegression(random_state=42, max_iter=1000)),
    'svm': make_pipeline(SVC(probability=True, random_state=42)),
    'naive_bayes': make_pipeline(GaussianNB()),
    'random_forest': make_pipeline(RandomForestClassifier(n_estimators=100, random_state=42)),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_metrics = {}
for name, model in models.items():
    probabilities = cross_val_predict(model, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
    threshold, metrics = choose_screening_threshold(y_train, probabilities)
    cv_metrics[name] = {'threshold': threshold, **metrics}

# Screening policy: meet the target sensitivity, then maximize specificity.
selected_key = max(models, key=lambda name: (
    cv_metrics[name]['specificity'],
    cv_metrics[name]['weighted_f1_score'],
    cv_metrics[name]['positive_recall'],
))
model_metrics = {}
for key, model in models.items():
    model.fit(X_train, y_train)
    threshold = cv_metrics[key]['threshold']
    test_probabilities = model.predict_proba(X_test)[:, 1]
    model_metrics[key] = {
        'threshold': threshold,
        'cv_metrics': cv_metrics[key],
        'test_metrics': calculate_metrics(y_test, test_probabilities, threshold),
    }

decision_config = {
    'model_key': selected_key,
    'model_name': MODEL_LABELS[selected_key],
    'target_sensitivity': TARGET_SENSITIVITY,
    'selection_metric': 'highest_specificity_subject_to_target_sensitivity',
    'selection_method': 'Per-model threshold selected from 5-fold out-of-fold train predictions to reach target positive recall; recommend highest CV specificity.',
    'model_keys': list(models),
    'models': model_metrics,
    'cv_sample_count': int(len(y_train)),
    'test_sample_count': int(len(y_test)),
}
model_path = output_dir / 'model.joblib'
config_path = output_dir / 'decision_config.json'
joblib.dump(models, model_path, compress=3)
config_path.write_text(json.dumps(decision_config, indent=2), encoding='utf-8')
print(f"Recommended: {MODEL_LABELS[selected_key]} (CV specificity={cv_metrics[selected_key]['specificity']:.4f}, threshold={cv_metrics[selected_key]['threshold']:.4f}, positive recall={cv_metrics[selected_key]['positive_recall']:.4f})")
print(f'Saved model.joblib and decision_config.json to {output_dir}')
if files is not None:
    files.download(str(model_path))
    files.download(str(config_path))
